# GR1T1 OSC Pose Controller Sweep
Run a wandb hyperparameter sweep on the GR1T1 `Lift` task to minimize end-effector pose error with the OSC pose controller utilities included in this repo.



## Notebook Overview
- Push your updated `robosuite` fork to GitHub, then set `REPO_URL` and `BRANCH` below.
- This notebook installs the repo in editable mode on Colab, sets up MuJoCo for headless rendering, and logs to wandb.
- Sweeps default to the pose targets stored in `osc_evals/ee_targets_2.json` and skip video uploads for faster runtime.
- Update the wandb project/entity to match your workspace before running.



In [1]:
# Install the desired Python version if not already present
!sudo apt-get update
!sudo apt-get install python3.10 python3.10-dev # Example for Python 3.8

# Configure alternatives
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.8 1
!sudo update-alternatives --config python3

# Verify the change
!python3 --version

Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (3,639 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (

In [7]:
!sudo apt update
!sudo apt install python3-pip

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (3,885 B/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
49 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of

In [8]:
import os

REPO_URL = "https://github.com/YOUR_USERNAME/robosuite.git"  # TODO: update to your fork
BRANCH = "main"
PROJECT = "osc-controller-evals"
ENTITY = "Sarthak-Dayal"  # e.g., "my-team"; set to None for personal wandb accounts
ROBOT = "GR1T1RightArmOnly"

os.environ.setdefault("MUJOCO_GL", "egl")

print(f"Repo: {REPO_URL}")
print(f"Branch: {BRANCH}")
print(f"wandb project: {PROJECT}")
print(f"wandb entity: {ENTITY}")
print(f"Robot: {ROBOT}")



Repo: https://github.com/YOUR_USERNAME/robosuite.git
Branch: main
wandb project: osc-controller-evals
wandb entity: Sarthak-Dayal
Robot: GR1T1RightArmOnly


In [14]:
%pip install -r /content/robosuite/requirements.txt

Obtaining file:///content/robosuite/robosuite (from -r /content/robosuite/requirements.txt (line 1))
ERROR: file:///content/robosuite/robosuite (from -r /content/robosuite/requirements.txt (line 1)) does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [4]:
import os
import pathlib
import subprocess
import sys

repo_name = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
if not pathlib.Path(repo_name).exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL], check=True)
os.chdir(repo_name)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb", "mujoco"], check=True)



CalledProcessError: Command '['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-e', '.']' returned non-zero exit status 1.

In [ ]:
import wandb
wandb.login()



In [ ]:
import json
from osc_evals.empirical.gr1t1_sweep import default_search_space

sweep_config = default_search_space(metric="overall_position_error")
sweep_config["name"] = "gr1t1_pose_controller"
print(json.dumps(sweep_config, indent=2))



In [ ]:
sweep_id = wandb.sweep(sweep=sweep_config, project=PROJECT, entity=ENTITY)
print("Sweep ID:", sweep_id)



In [ ]:
from osc_evals.empirical.gr1t1_sweep import run_gr1t1_pose_trial


def sweep_runner():
    with wandb.init(project=PROJECT, entity=ENTITY) as run:
        trial_config = dict(run.config)
        results = run_gr1t1_pose_trial(
            trial_config,
            wandb_project=PROJECT,
            wandb_entity=ENTITY,
            log_videos=False,
            robot_name=ROBOT,
        )
        if results:
            numeric_results = {k: v for k, v in results.items() if isinstance(v, (int, float))}
            if numeric_results:
                wandb.log(numeric_results)
        print("Completed run with results:", results)



In [ ]:
SWEEP_TRIALS = 10
wandb.agent(sweep_id, function=sweep_runner, count=SWEEP_TRIALS)



### Notes
- Each agent run reuses the shared wandb run created inside `sweep_runner`, so the empirical test will not re-initialize wandb and video logging stays off by default.
- The helper defaults to `GR1T1RightArmOnly` to match the single-arm controller wiring; override `ROBOT` if you add support for other variants.
- To evaluate a single configuration outside the sweep, call `run_gr1t1_pose_trial({...})` directly (optionally set `log_videos=True` once you have promising candidates).
- Review the `overall_position_error` metric in the wandb dashboard to pick the best controller gains, then re-run the helper with that config to record higher-quality videos if needed.

